# MP29 — Transformer Encoder Block

Combine attention, normalization, residual paths and a feed-forward network. [Brief](../stage_07_transformers/MP29_transformer_encoder_block/README.md) · [Validation](../stage_07_transformers/MP29_transformer_encoder_block/VALIDATION.md). Prerequisite: MP28.

This is an untrained pre-normalized encoder, not a full paper reproduction. CPU float64, dropout0, no downloads or optimization. The measured crop is training-only NAIP aerial RGB, fixed DN/10000 with unverified physical calibration.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
import torch
import matplotlib.pyplot as plt
root=Path.cwd() if (Path.cwd()/'src/rssr').exists() else Path.cwd().parent
sys.path.insert(0,str(root/'stage_07_transformers/MP29_transformer_encoder_block'))
from encoder_tools import Encoder,positions
torch.set_num_threads(1);torch.manual_seed(29)


## 1. Equations, dimensions and normalization

For X[B,N,D], set Z=X+PE (or Z=X without positions). Pre-normalization means Y=Z+MHA(LN₁(Z)), then O=Y+W₂ GELU(W₁ LN₂(Y)+b₁)+b₂. LN operates over D features independently for each token using population variance, epsilon10⁻⁵ and learned gain/bias. FFN changes D→2D→D. Residual addition requires matching shapes.

For features[1,3], mean=2 and population variance=1, so normalized values are[−1,1]/√(1+10⁻⁵), approximately[−0.999995,0.999995]. This differs from sample variance and from normalization across tokens.

In [ ]:
block=Encoder(2,1).double()
actual=block.norm1(torch.tensor([[[1.,3.]]],dtype=torch.float64))
expected=torch.tensor([[[-1.,1.]]],dtype=torch.float64)/(1+1e-5)**.5
torch.testing.assert_close(actual,expected)
print(actual.detach().numpy())


## 2. Encoder diagram and actual shape trace

At B=1,N=16,D=4 the attention branch retains[1,16,4], the feed-forward hidden tensor is[1,16,8], and the output is[1,16,4]. There is no spatial resizing in this block. Residual paths carry the original representation even if their learned branches output zero.

In [ ]:
block=Encoder(4,2).double();x=torch.ones(1,16,4,dtype=torch.float64)
out,trace=block(x)
print(json.dumps(trace,indent=2))
assert trace['ffn_hidden']==[1,16,8] and trace['output']==[1,16,4]
fig,ax=plt.subplots(figsize=(11,3));ax.set(xlim=(-.5,5.5),ylim=(-.6,1.2));ax.axis('off')
labels=['X + optional PE','LN → MHA','Add X','LN → FFN','Add Y','Output']
for i,label in enumerate(labels):
    ax.text(i,0,label,ha='center',va='center',bbox=dict(boxstyle='round,pad=.5',facecolor='#dceef4'))
    if i<5:ax.annotate('',xy=(i+.65,0),xytext=(i+.4,0),arrowprops=dict(arrowstyle='->'))
ax.annotate('',xy=(2,.3),xytext=(0,.3),arrowprops=dict(arrowstyle='->',connectionstyle='arc3,rad=-.4'))
ax.annotate('',xy=(4,.3),xytext=(2,.3),arrowprops=dict(arrowstyle='->',connectionstyle='arc3,rad=-.4'))
ax.text(2.5,-.45,'Token shape stays [B,N,D]; FFN hidden width is 2D',ha='center')
fig.tight_layout();plt.show()


## 3. What positions change

For feature pair2i,2i+1 use sin(p/10000^(2i/D)), cos(p/10000^(2i/D)). At p=0 the width4 encoding is[0,1,0,1]. The implementation requires even width. Without positions, a token permutation yields the same output permutation. Adding encodings tied to fixed slots breaks that property when token contents are rearranged but slots are not.

In [ ]:
torch.manual_seed(29);block=Encoder().double().eval()
x=torch.randn(1,5,4,dtype=torch.float64);p=[3,0,4,1,2]
with torch.no_grad():
    y,_=block(x);yp,_=block(x[:,p]);torch.testing.assert_close(yp,y[:,p])
    located,_=block(x+positions(5,4));moved,_=block(x[:,p]+positions(5,4))
print('Fixed-slot positional difference:',float((moved-located[:,p]).abs().max()))
assert not torch.allclose(moved,located[:,p])


## 4. Measured fixed-weight response comparison

The twelve recorded conditions combine two inputs (Gaussian and a pinned4×4 NAIP training crop),1/2/4 heads and positions off/on. Width4 and every learned parameter are held identical across head counts. The RGB→width4 random embedding is also fixed. No model is trained.

Plot RMS of the block's change relative to its actual input, not distance to a target image. Magnitude is not quality. Positional encodings are order-one while the numerical RGB values are small, so their effect is scale-dependent.

In [ ]:
record=json.loads((root/'reports/mp29-encoder.json').read_text())
assert record['source']['split']=='train' and all(r['finite'] for r in record['rows'])
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,name in zip(axes,['synthetic_normal','naip_train_rgb']):
    for positional in [False,True]:
        rows=[r for r in record['rows'] if r['input']==name and r['positions']==positional]
        ax.plot([r['heads'] for r in rows],[r['residual_change_rms'] for r in rows],'o-',label='positions on' if positional else 'positions off')
    ax.set(title=name,xlabel='Head count (fixed width4)',ylabel='RMS block change (feature units)',xticks=[1,2,4]);ax.legend();ax.grid(alpha=.2)
fig.tight_layout();plt.show()
print('Parameters per encoder:',sorted(set(r['parameters'] for r in record['rows'])))


## 5. Residual and gradient sanity checks

If both branches are zero, output equals input exactly. At a constant token LN has zero variance, but epsilon keeps its denominator nonzero. Backpropagation must remain finite; this does not establish healthy gradients for arbitrary trained depth.

In [ ]:
zero=Encoder().double()
with torch.no_grad():
    for parameter in zero.parameters():parameter.zero_()
sample=torch.randn(1,3,4,dtype=torch.float64);out,_=zero(sample);torch.testing.assert_close(out,sample)
normal=Encoder().double();constant=torch.ones(1,3,4,dtype=torch.float64,requires_grad=True)
out,_=normal(constant);out.square().mean().backward()
assert torch.isfinite(constant.grad).all()
print('Zero branches preserve residual; constant-token gradients finite')


## Practice and separate solutions

Predict the shape after each residual, calculate LN([1,3]), and explain why output RMS cannot select an SR model. Then compare with the trace and hand solution above. Residuals stay[B,N,D]; RMS has no HR-reference comparison and can reward arbitrary scaling.

Reflection: pre-normalization exposes clear residual routes, while positions alter order sensitivity. A failed shape or nonfinite input is rejected explicitly. Next: patchification and window connectivity (MP30). Limitations: no training, positional ablation on a learned task, masking, dropout or dataset generalization. Learner mastery unassessed.

Reference: Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762). This pre-LN educational variant differs from the original post-LN encoder ordering.